# Verify FDG Stage Profile JIT CPU/GPU

This notebook profiles the flexible dual grid implementation stage by stage:

1. Choose whether to JIT compile the current local sources or directly import an already-installed `o_voxel` package.
2. Load the selected Python API.
3. Generate synthetic Gaussian triangle soup directly in the notebook, including a few degenerate triangles.
4. Time `intersect_qef`, `face_qef`, `boundary_qef`, and the full `mesh_to_flexible_dual_grid` pipeline on both CPU and GPU.
5. Report the residual runtime between the summed three stages and the full pipeline.

No local mesh file is loaded.


In [1]:
import os
import sys
import time
import types
import importlib
import importlib.util
from pathlib import Path

import numpy as np
import torch
from torch.utils.cpp_extension import load

import pandas as pd


In [2]:
ROOT = Path(r'/mnt/nvmefs/Projects/Part Generation/TRELLIS.2-o-voxel-gpu-mod/o-voxel').resolve()
USE_JIT = False
INSTALLED_IMPORT_NAME = 'o_voxel'

print('ROOT =', ROOT)
print('USE_JIT =', USE_JIT)
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device =', torch.cuda.get_device_name(0))


def build_jit_extension():
    sources = [
    'src/hash/hash.cu',
    'src/convert/flexible_dual_grid.cpp',
    'src/convert/volumetic_attr.cpp',
    'src/convert/mesh_to_flexible_dual_grid_gpu/torch_bindings.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/flexible_dual_grid_gpu.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/intersection_qef.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/voxelize_mesh_oct.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/voxel_traverse_edge_dda.cu',
    'src/serialize/api.cu',
    'src/serialize/hilbert.cu',
    'src/serialize/z_order.cu',
    'src/io/svo.cpp',
    'src/io/filter_parent.cpp',
    'src/io/filter_neighbor.cpp',
    'src/rasterize/rasterize.cu',
    'src/ext.cpp',
]
    full_sources = [str(ROOT / s) for s in sources]
    missing = [s for s in full_sources if not Path(s).exists()]
    if missing:
        raise FileNotFoundError(f'Missing sources: {missing}')

    build_dir = ROOT / '.verify_build'
    build_dir.mkdir(parents=True, exist_ok=True)

    unique_suffix = f"{os.getpid()}_{time.time_ns()}_{os.urandom(4).hex()}"
    mod_name = f"o_voxel_verify_{unique_suffix}"

    max_jobs = max(1, os.cpu_count() or 1)
    os.environ['MAX_JOBS'] = str(max_jobs)
    print('MAX_JOBS =', os.environ['MAX_JOBS'])
    print('JIT module name =', mod_name)

    ext_mod = load(
        name=mod_name,
        sources=full_sources,
        extra_include_paths=[str(ROOT / 'third_party/eigen')],
        extra_cflags=['-O3', '-std=c++17'],
        extra_cuda_cflags=['-O3', '-std=c++17', '--expt-relaxed-constexpr'],
        with_cuda=True,
        build_directory=str(build_dir),
        verbose=True,
    )
    print('JIT build/link: OK')
    print('jit module path =', ext_mod.__file__)
    return ext_mod


def load_local_flexible_dual_grid(ext_mod):
    pkg = types.ModuleType('o_voxel')
    pkg.__path__ = [str(ROOT / 'o_voxel')]
    pkg._C = ext_mod
    sys.modules['o_voxel'] = pkg
    sys.modules['o_voxel._C'] = ext_mod

    convert_pkg = types.ModuleType('o_voxel.convert')
    convert_pkg.__path__ = [str(ROOT / 'o_voxel' / 'convert')]
    sys.modules['o_voxel.convert'] = convert_pkg

    spec = importlib.util.spec_from_file_location(
        'o_voxel.convert.flexible_dual_grid',
        ROOT / 'o_voxel' / 'convert' / 'flexible_dual_grid.py',
    )
    mod = importlib.util.module_from_spec(spec)
    sys.modules['o_voxel.convert.flexible_dual_grid'] = mod
    spec.loader.exec_module(mod)
    return mod


if USE_JIT:
    ext_mod = build_jit_extension()
    fdg_api = load_local_flexible_dual_grid(ext_mod)
    api_mode = 'jit'
else:
    installed_pkg = importlib.import_module(INSTALLED_IMPORT_NAME)
    ext_mod = installed_pkg._C
    fdg_api = importlib.import_module(f'{INSTALLED_IMPORT_NAME}.convert.flexible_dual_grid')
    api_mode = 'installed'
    print('installed package path =', getattr(installed_pkg, '__file__', '<package>'))
    print('installed extension path =', getattr(ext_mod, '__file__', '<extension>'))
    print('installed API path =', getattr(fdg_api, '__file__', '<module>'))

print('api_mode =', api_mode)
print('ext_mod =', ext_mod)
print('fdg_api =', fdg_api)


ROOT = /mnt/nvmefs/Projects/Part Generation/TRELLIS.2-o-voxel-gpu-mod/o-voxel
USE_JIT = False
torch = 2.6.0+cu124
cuda available = True
cuda device = NVIDIA GeForce RTX 4090
installed package path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/__init__.py
installed extension path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/_C.cpython-310-x86_64-linux-gnu.so
installed API path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/convert/flexible_dual_grid.py
api_mode = installed
ext_mod = <module 'o_voxel._C' from '/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/_C.cpython-310-x86_64-linux-gnu.so'>
fdg_api = <module 'o_voxel.convert.flexible_dual_grid' from '/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/convert/flexible_dual_grid.py'>


In [3]:
print('has mesh_to_flexible_dual_grid_cpu =', hasattr(ext_mod, 'mesh_to_flexible_dual_grid_cpu'))
print('has mesh_to_flexible_dual_grid_gpu =', hasattr(ext_mod, 'mesh_to_flexible_dual_grid_gpu'))
print('has intersection_occ =', hasattr(fdg_api, 'intersection_occ'))
print('has intersect_qef =', hasattr(fdg_api, 'intersect_qef'))
print('has voxelize_mesh_gpu =', hasattr(fdg_api, 'voxelize_mesh_gpu'))
print('has voxelize_edge_gpu =', hasattr(fdg_api, 'voxelize_edge_gpu'))
print('has face_qef =', hasattr(fdg_api, 'face_qef'))
print('has voxel_traverse_edge_dda_gpu =', hasattr(fdg_api, 'voxel_traverse_edge_dda_gpu'))
print('has boundary_qef =', hasattr(fdg_api, 'boundary_qef'))


has mesh_to_flexible_dual_grid_cpu = True
has mesh_to_flexible_dual_grid_gpu = True
has intersection_occ = True
has intersect_qef = True
has voxelize_mesh_gpu = True
has voxelize_edge_gpu = True
has face_qef = True
has voxel_traverse_edge_dda_gpu = True
has boundary_qef = True


In [4]:
assert torch.cuda.is_available(), 'CUDA is required for this notebook.'
device = torch.device('cuda:0')
torch.cuda.set_device(device)

GRID = 128
N_VERT = 12000
N_FACE = 36000
FACE_WEIGHT = 1.0
BOUNDARY_WEIGHT = 0.2
REGULARIZATION_WEIGHT = 1e-2
INTERSECT_CHUNK_TRIANGLES = 262144
BOUNDARY_CHUNK_STEPS = 1024
WARMUP = 3
ITERS = 10
AABB = [[0.0, 0.0, 0.0], [1.0, 1.0, 1.0]]

torch.manual_seed(13)
vertices_gpu = (0.5 + 0.18 * torch.randn(N_VERT, 3, device=device, dtype=torch.float32)).clamp_(0.0, 1.0)
vertices_gpu[1] = vertices_gpu[0]
vertices_gpu[3] = vertices_gpu[2]
faces_gpu = torch.randint(0, N_VERT, (N_FACE, 3), device=device, dtype=torch.int64)
faces_gpu[:3] = torch.tensor([[0, 0, 1], [2, 3, 3], [4, 4, 4]], device=device, dtype=torch.int64)
faces_gpu = faces_gpu.to(torch.int32).contiguous()

vertices_cpu = vertices_gpu.cpu().contiguous()
faces_cpu = faces_gpu.cpu().contiguous()

aabb_gpu = torch.tensor(AABB, dtype=torch.float32, device=device)
aabb_cpu = aabb_gpu.cpu()
voxel_size_gpu = ((aabb_gpu[1] - aabb_gpu[0]) / GRID).to(torch.float32).contiguous()
voxel_size_cpu = voxel_size_gpu.cpu().contiguous()
grid_range_gpu = torch.tensor([[0, 0, 0], [GRID, GRID, GRID]], dtype=torch.int32, device=device)
grid_range_cpu = grid_range_gpu.cpu().contiguous()

triangles_gpu = vertices_gpu[faces_gpu.long()].contiguous().to(torch.float32)
triangles_cpu = triangles_gpu.cpu().contiguous()

print('vertices_gpu:', vertices_gpu.shape)
print('faces_gpu:', faces_gpu.shape)
print('triangles_gpu:', triangles_gpu.shape)
print('voxel_size_gpu:', voxel_size_gpu)
print('grid_range_gpu:', grid_range_gpu)


vertices_gpu: torch.Size([12000, 3])
faces_gpu: torch.Size([36000, 3])
triangles_gpu: torch.Size([36000, 3, 3])
voxel_size_gpu: tensor([0.0078, 0.0078, 0.0078], device='cuda:0')
grid_range_gpu: tensor([[  0,   0,   0],
        [128, 128, 128]], device='cuda:0', dtype=torch.int32)


In [5]:
def build_boundary_segments(vertices: torch.Tensor, faces: torch.Tensor):
    faces_i64 = faces.to(torch.int64).cpu()
    e01 = faces_i64[:, [0, 1]]
    e12 = faces_i64[:, [1, 2]]
    e20 = faces_i64[:, [2, 0]]
    edges = torch.cat([e01, e12, e20], dim=0)
    edges = torch.sort(edges, dim=1).values
    unique_edges, counts = torch.unique(edges, dim=0, return_counts=True)
    boundary_edges = unique_edges[counts == 1]
    boundary_edges = boundary_edges.to(device=vertices.device, dtype=torch.long)
    return vertices[boundary_edges].contiguous().to(torch.float32)

boundaries_gpu = build_boundary_segments(vertices_gpu, faces_gpu)
boundaries_cpu = boundaries_gpu.cpu().contiguous()
print('boundaries_gpu:', boundaries_gpu.shape)


boundaries_gpu: torch.Size([107821, 2, 3])


In [6]:
def time_call(fn, warmup=WARMUP, iters=ITERS, sync_device=None):
    out = None
    for _ in range(warmup):
        out = fn()
        if sync_device is not None:
            torch.cuda.synchronize(sync_device)

    times_ms = []
    for _ in range(iters):
        if sync_device is not None:
            torch.cuda.synchronize(sync_device)
        t0 = time.perf_counter()
        out = fn()
        if sync_device is not None:
            torch.cuda.synchronize(sync_device)
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)
    return out, np.asarray(times_ms, dtype=np.float64)


def summarize_times(device_name, stage_name, times_ms):
    return {
        'device': device_name,
        'stage': stage_name,
        'mean_ms': float(times_ms.mean()),
        'std_ms': float(times_ms.std(ddof=0)),
        'min_ms': float(times_ms.min()),
        'max_ms': float(times_ms.max()),
    }


In [7]:
(voxels_gpu, mean_sum_gpu, cnt_gpu, intersected_gpu, qefs_gpu), t_intersect_gpu = time_call(
    lambda: fdg_api.intersect_qef(
        triangles_gpu,
        voxel_size_gpu,
        grid_range_gpu,
        chunk_triangles=INTERSECT_CHUNK_TRIANGLES,
    ),
    sync_device=device,
)
(voxels_cpu, mean_sum_cpu, cnt_cpu, intersected_cpu, qefs_cpu), t_intersect_cpu = time_call(
    lambda: fdg_api.intersect_qef(
        triangles_cpu,
        voxel_size_cpu,
        grid_range_cpu,
        chunk_triangles=INTERSECT_CHUNK_TRIANGLES,
    ),
)
print("intersect_qef gpu/cpu voxel counts:", voxels_gpu.shape[0], voxels_cpu.shape[0])

intersect_qef gpu/cpu voxel counts: 1118854 1118854


In [8]:
qefs_face_gpu, t_face_gpu = time_call(
    lambda: fdg_api.face_qef(triangles_gpu, voxel_size_gpu, grid_range_gpu, voxels_gpu),
    sync_device=device,
)
qefs_face_cpu, t_face_cpu = time_call(
    lambda: fdg_api.face_qef(triangles_cpu, voxel_size_cpu, grid_range_cpu, voxels_cpu),
)
print('face_qef gpu/cpu shapes:', qefs_face_gpu.shape, qefs_face_cpu.shape)


face_qef gpu/cpu shapes: torch.Size([1118854, 4, 4]) torch.Size([1118854, 4, 4])


In [9]:
qefs_boundary_gpu, t_boundary_gpu = time_call(
    lambda: fdg_api.boundary_qef(boundaries_gpu, voxel_size_gpu, grid_range_gpu, BOUNDARY_WEIGHT, voxels_gpu, chunk_steps=BOUNDARY_CHUNK_STEPS),
    sync_device=device,
)
qefs_boundary_cpu, t_boundary_cpu = time_call(
    lambda: fdg_api.boundary_qef(boundaries_cpu, voxel_size_cpu, grid_range_cpu, BOUNDARY_WEIGHT, voxels_cpu, chunk_steps=BOUNDARY_CHUNK_STEPS),
)
print('boundary_qef gpu/cpu shapes:', qefs_boundary_gpu.shape, qefs_boundary_cpu.shape)


boundary_qef gpu/cpu shapes: torch.Size([1118854, 4, 4]) torch.Size([1118854, 4, 4])


In [10]:
full_gpu, t_full_gpu = time_call(
    lambda: fdg_api.mesh_to_flexible_dual_grid(
        vertices_gpu,
        faces_gpu,
        grid_size=GRID,
        aabb=AABB,
        face_weight=FACE_WEIGHT,
        boundary_weight=BOUNDARY_WEIGHT,
        regularization_weight=REGULARIZATION_WEIGHT,
        intersect_chunk_triangles=INTERSECT_CHUNK_TRIANGLES,
        boundary_chunk_steps=BOUNDARY_CHUNK_STEPS,
    ),
    sync_device=device,
)
full_cpu, t_full_cpu = time_call(
    lambda: fdg_api.mesh_to_flexible_dual_grid(
        vertices_cpu,
        faces_cpu,
        grid_size=GRID,
        aabb=AABB,
        face_weight=FACE_WEIGHT,
        boundary_weight=BOUNDARY_WEIGHT,
        regularization_weight=REGULARIZATION_WEIGHT,
        timing=False,
    ),
)
print('full gpu voxel count:', full_gpu[0].shape[0])
print('full cpu voxel count:', full_cpu[0].shape[0])


full gpu voxel count: 1118854
full cpu voxel count: 1118854


In [11]:
rows = [
    summarize_times('gpu', 'intersect_qef', t_intersect_gpu),
    summarize_times('cpu', 'intersect_qef', t_intersect_cpu),
    summarize_times('gpu', 'face_qef', t_face_gpu),
    summarize_times('cpu', 'face_qef', t_face_cpu),
    summarize_times('gpu', 'boundary_qef', t_boundary_gpu),
    summarize_times('cpu', 'boundary_qef', t_boundary_cpu),
    summarize_times('gpu', 'full_fdg', t_full_gpu),
    summarize_times('cpu', 'full_fdg', t_full_cpu),
]

stage_df = pd.DataFrame(rows)
stage_df


,device,stage,mean_ms,std_ms,min_ms,max_ms
0,gpu,intersect_qef,1281.980305,16.990325,1257.201176,1312.670635
1,cpu,intersect_qef,10661.798837,398.574779,9799.409370,11291.610871
2,gpu,face_qef,315.729265,1.217875,314.277277,317.836051
3,cpu,face_qef,8524.037188,153.039298,8334.895461,8864.845227
4,gpu,boundary_qef,58.098825,1.007474,57.125115,60.802159
5,cpu,boundary_qef,851.024293,3.636328,843.272538,855.898882
6,gpu,full_fdg,1551.464515,17.760516,1533.126435,1582.916710
7,cpu,full_fdg,21210.323991,452.196012,20212.203654,21949.401679


In [12]:
summary = pd.DataFrame([
    {
        'device': 'gpu',
        'three_stage_sum_mean_ms': float(t_intersect_gpu.mean() + t_face_gpu.mean() + t_boundary_gpu.mean()),
        'full_mean_ms': float(t_full_gpu.mean()),
        'residual_mean_ms': float(t_full_gpu.mean() - (t_intersect_gpu.mean() + t_face_gpu.mean() + t_boundary_gpu.mean())),
    },
    {
        'device': 'cpu',
        'three_stage_sum_mean_ms': float(t_intersect_cpu.mean() + t_face_cpu.mean() + t_boundary_cpu.mean()),
        'full_mean_ms': float(t_full_cpu.mean()),
        'residual_mean_ms': float(t_full_cpu.mean() - (t_intersect_cpu.mean() + t_face_cpu.mean() + t_boundary_cpu.mean())),
    },
])
summary


,device,three_stage_sum_mean_ms,full_mean_ms,residual_mean_ms
0,gpu,1655.808395,1551.464515,-104.343879
1,cpu,20036.860317,21210.323991,1173.463674


In [13]:
speedup_df = pd.DataFrame([
    {
        'stage': 'intersect_qef',
        'cpu_over_gpu': float(t_intersect_cpu.mean() / t_intersect_gpu.mean()),
    },
    {
        'stage': 'face_qef',
        'cpu_over_gpu': float(t_face_cpu.mean() / t_face_gpu.mean()),
    },
    {
        'stage': 'boundary_qef',
        'cpu_over_gpu': float(t_boundary_cpu.mean() / t_boundary_gpu.mean()),
    },
    {
        'stage': 'full_fdg',
        'cpu_over_gpu': float(t_full_cpu.mean() / t_full_gpu.mean()),
    },
])
speedup_df


,stage,cpu_over_gpu
0,intersect_qef,8.316664
1,face_qef,26.997932
2,boundary_qef,14.647874
3,full_fdg,13.671163
